In [1]:
#This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-

In [2]:
# %% [markdown]
# ARC-AGI-3 Wynt3r45 Meta-Agent
# License: CC BY 4.0 (or submission-appropriate equivalent)
# References:
# - https://docs.arcprize.org/llms.txt
# - ARC-AGI-3 Toolkit / ARC-AGI-3 Agents docs
# - ARC Prize 2026 / ARC-AGI-3 competition materials
#
# This notebook supports:
# - Online: docs discovery, API games, scorecards, replays, REST control
# - Offline: Kaggle input discovery, local file handling, toolkit/local environment
# - Toolkit: Arcade, EnvironmentWrapper, render modes, scorecards, recordings
# - REST API: RESET, ACTION1-7, scorecards open/get/close, per-game retrieval
# - Agent layer: Wynt3r45 persona on top of ARC-AGI-3 stack

# %% [code]
import os
import json
import time
import random
import pathlib
import subprocess
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import requests

try:
    import kagglehub
except Exception:
    kagglehub = None

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from arc_agi import Arcade, OperationMode
except Exception:
    Arcade = None
    OperationMode = None

try:
    from arcengine import GameAction, GameState
except Exception:
    GameAction = None
    GameState = None

try:
    import agentops
except Exception:
    agentops = None

DOC_INDEX_URL = "https://docs.arcprize.org/llms.txt"
ROOT_URL = "https://three.arcprize.org"
DEFAULT_GAME = "ls20"

# %% [code]
# Kaggle environment discovery (exact snippet requested)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# %% [code]
# Optional: install-time notes for a fresh notebook environment
# If needed, uncomment:
# !pip install -q requests python-dotenv kagglehub
# !pip install -q arc-agi agentops

# %% [code]
def ensure_env():
    if load_dotenv is not None:
        load_dotenv(".env")

    # Do not overwrite existing keys unless you explicitly want to.
    # Example patterns requested by you:
    # os.environ["ARC_API_KEY"] = "b111d3aa-8be6-4dcf-8883-fbb24a36f3ad"
    # with open(".env", "w") as f:
    #     f.write('ARC_API_KEY="b111d3aa-8be6-4dcf-8883-fbb24a36f3ad"')

    return os.getenv("ARC_API_KEY")

API_KEY = ensure_env()

# %% [code]
def internet_check() -> bool:
    """
    Tries the docs index first because that is the canonical online discovery source.
    If that fails, optionally fall back to a small HTTP check.
    """
    try:
        r = requests.get(DOC_INDEX_URL, timeout=5)
        return r.ok
    except Exception:
        return False

def ping_pypi_heuristic() -> Tuple[bool, str]:
    """
    Kaggle/offline heuristic requested by you.
    Note: ping may fail even when some internet is present, so treat this as heuristic only.
    """
    try:
        result = subprocess.run(
            ["bash", "-lc", "ping -c 1 pypi.org"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        ok = result.returncode == 0
        return ok, (result.stdout + "\n" + result.stderr).strip()
    except Exception as e:
        return False, str(e)

ONLINE_AVAILABLE = internet_check()

# %% [code]
def fetch_docs_index(online: bool = True) -> str:
    """
    Pulls docs from the online index when online is available.
    Offline fallback tries local cached docs if you place them in /kaggle/input or cwd.
    """
    if online:
        try:
            return requests.get(DOC_INDEX_URL, timeout=10).text
        except Exception as e:
            return f"[DOCS_UNAVAILABLE_ONLINE] {e}"

    candidates = [
        pathlib.Path("/kaggle/input/arc_docs/llms.txt"),
        pathlib.Path("./llms.txt"),
        pathlib.Path("./docs/llms.txt"),
    ]
    for p in candidates:
        if p.exists():
            return p.read_text(encoding="utf-8", errors="ignore")

    return "[DOCS_UNAVAILABLE_OFFLINE] No local llms.txt cache found."

# %% [code]
DOC_HINTS = [
    "quickstart",
    "api-keys",
    "vocabulary",
    "create-agent",
    "llm_agents",
    "local-vs-online",
    "scorecards",
    "recordings",
    "swarms",
    "benchmarking-agent",
    "competition-mode",
    "listen-and-serve",
    "render-games",
    "full-play-test",
    "feature-requests-bugs",
    "contributing",
    "changelog",
    "partner_templates/agentops",
    "partner_templates/anthropic",
    "partner_templates/huggingface",
    "partner_templates/langchain",
]

# %% [code]
def kaggle_input_files() -> List[str]:
    files = []
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            files.append(os.path.join(dirname, filename))
    return files

def print_kaggle_input_files():
    for path in kaggle_input_files():
        print(path)

# %% [code]
def download_competition_data() -> Dict[str, Any]:
    """
    Tries KaggleHub first, then CLI guidance, then MCP reminder.
    """
    result: Dict[str, Any] = {"status": "unknown", "path": None, "notes": []}

    if kagglehub is not None:
        try:
            path = kagglehub.competition_download("arc-prize-2026-arc-agi-3")
            result["status"] = "kagglehub_ok"
            result["path"] = str(path)
            return result
        except Exception as e:
            result["notes"].append(f"kagglehub failed: {e}")

    # Kaggle CLI fallback guidance (works when available in the notebook shell)
    result["notes"].append(
        "CLI fallback: kaggle competitions download -c arc-prize-2026-arc-agi-3"
    )

    # MCP tool reminder requested by you
    result["notes"].append(
        'MCP tool: prompt your client to use "mcp_kaggle_download_competition_data_files"'
    )

    return result

# %% [code]
@dataclass
class Wynt3r45Config:
    run_mode: str = field(default_factory=lambda: os.getenv("ARC_RUN_MODE", "auto").lower())
    api_key: str = field(default_factory=lambda: os.getenv("ARC_API_KEY", ""))
    base_url: str = ROOT_URL
    game_id: str = DEFAULT_GAME
    scorecard_tags: List[str] = field(default_factory=lambda: ["wynt3r45", "arc-agi-3", "kaggle"])
    source_url: str = "https://github.com/example/agent"
    opaque: Dict[str, Any] = field(default_factory=lambda: {"model": "wynt3r45", "temperature": 0.25})
    recordings_dir: str = "./recordings"
    environments_dir: str = "./environment_files"

    def choose_mode(self) -> str:
        if self.run_mode in {"online", "offline"}:
            return self.run_mode
        return "online" if ONLINE_AVAILABLE and bool(self.api_key) else "offline"

CFG = Wynt3r45Config()

# %% [code]
class ArcRestClient:
    def __init__(self, api_key: str, base_url: str = ROOT_URL):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.session = requests.Session()
        self.session.headers.update({
            "X-API-Key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
        })

    def _get(self, path: str, **kwargs):
        r = self.session.get(f"{self.base_url}{path}", **kwargs)
        r.raise_for_status()
        return r

    def _post(self, path: str, payload: Dict[str, Any], **kwargs):
        r = self.session.post(f"{self.base_url}{path}", json=payload, **kwargs)
        r.raise_for_status()
        return r

    def list_games(self):
        return self._get("/api/games").json()

    def open_scorecard(self, source_url=None, tags=None, opaque=None):
        # exact user-provided payload pattern included
        payload = {
            "source_url": source_url or CFG.source_url,
            "tags": tags or CFG.scorecard_tags,
            "opaque": opaque or CFG.opaque,
        }
        r = self._post("/api/scorecard/open", payload)
        return r.json()["card_id"], r.text

    def get_scorecard(self, card_id):
        return self._get(f"/api/scorecard/{card_id}").json()

    def get_scorecard_game(self, card_id, game_id):
        return self._get(f"/api/scorecard/{card_id}/{game_id}").json()

    def close_scorecard(self, card_id):
        return self._post("/api/scorecard/close", {"card_id": card_id}).json()

    def reset(self, game_id, card_id, guid=None):
        payload = {"game_id": game_id, "card_id": card_id}
        if guid is not None:
            payload["guid"] = guid
        return self._post("/api/cmd/RESET", payload).json()

    def action(self, n: int, game_id, guid, reasoning=None, x=None, y=None):
        payload = {"game_id": game_id, "guid": guid}
        if reasoning is not None:
            payload["reasoning"] = reasoning
        if n == 6:
            if x is None or y is None:
                x, y = 32, 32
            payload["x"] = int(x)
            payload["y"] = int(y)
        return self._post(f"/api/cmd/ACTION{n}", payload).json()

# %% [code]
def open_scorecard_exact_example(api_key: str):
    import requests

    url = "https://three.arcprize.org/api/scorecard/open"
    payload = {
        "source_url": "https://github.com/example/agent",
        "tags": ["baseline", "gpt-4o"],
        "opaque": {
            "model": "gpt-4o",
            "temperature": 0.25
        }
    }
    headers = {
        "X-API-Key": api_key,
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)
    print(response.text)
    return response

# %% [code]
def list_games_online(client: ArcRestClient):
    return client.list_games()

def list_games_toolkit(arc):
    envs = arc.get_environments()
    return [{"game_id": e.game_id, "title": e.title, "tags": getattr(e, "tags", [])} for e in envs]

# %% [code]
def choose_best_mode() -> str:
    return CFG.choose_mode()

MODE = choose_best_mode()
MODE

# %% [code]
class Wynt3r45Persona:
    def __init__(self):
        self.memory = []
        self.notes = []

    def analyze_frame(self, frame):
        # Minimal, safe placeholder for fluid-intelligence style reasoning
        return {
            "pattern": "unknown",
            "symmetry_hint": False,
            "center_bias": True,
            "confidence": 0.55,
        }

    def choose_action(self, available_actions: List[int], state: Optional[Dict[str, Any]] = None):
        available_actions = list(available_actions or [1, 2, 3, 4, 5, 6, 7])
        if 6 in available_actions:
            return 6
        if 7 in available_actions and len(self.memory) > 4:
            return 7
        return random.choice(available_actions)

    def build_reasoning(self, analysis: Dict[str, Any]):
        return {
            "persona": "Wynt3r45",
            "thought": "Observe -> infer -> act -> learn",
            "analysis": analysis,
        }

    def remember(self, entry):
        self.memory.append(entry)

PERSONA = Wynt3r45Persona()

# %% [code]
def run_api_full_playtest(game_id: Optional[str] = None, max_steps: int = 50):
    """
    Full play test: list games, open scorecard, RESET, ACTIONS, close scorecard.
    """
    if not CFG.api_key:
        raise RuntimeError("ARC_API_KEY is required for online/API mode.")

    client = ArcRestClient(CFG.api_key, CFG.base_url)
    games = client.list_games()
    if not games:
        raise RuntimeError("No API games available.")

    selected_game = game_id or games[0]["game_id"]

    card_id, raw_open_response = client.open_scorecard(
        source_url=CFG.source_url,
        tags=CFG.scorecard_tags,
        opaque=CFG.opaque,
    )

    print("Opened scorecard:", card_id)
    print("Open response text:", raw_open_response)

    state = client.reset(selected_game, card_id)
    guid = state["guid"]

    for step in range(max_steps):
        if state["state"] in {"WIN", "GAME_OVER", "NOT_STARTED"}:
            break

        analysis = PERSONA.analyze_frame(state.get("frame"))
        action = PERSONA.choose_action(state.get("available_actions", []), state)
        reasoning = PERSONA.build_reasoning(analysis)

        if action == 6:
            state = client.action(6, selected_game, guid, reasoning=reasoning, x=32, y=32)
        else:
            state = client.action(action, selected_game, guid, reasoning=reasoning)

        PERSONA.remember({"step": step, "action": action, "state": state.get("state")})
        print(f"[{step}] action={action} state={state.get('state')} score={state.get('score', 'n/a')}")

    final_summary = client.close_scorecard(card_id)
    return {
        "card_id": card_id,
        "final_summary": final_summary,
        "last_state": state,
    }

# %% [code]
def run_api_playtest_exactly_as_requested():
    """
    Your exact open-scorecard request pattern, wrapped in a runnable notebook function.
    """
    if not CFG.api_key:
        raise RuntimeError("Set ARC_API_KEY first.")

    return open_scorecard_exact_example(CFG.api_key)

# %% [code]
def run_toolkit_mode(game_id: str = DEFAULT_GAME, render_mode: Optional[str] = "terminal", save_recording: bool = True):
    """
    Toolkit + EnvironmentWrapper mode: local/offline when available, or online via toolkit.
    """
    if Arcade is None or OperationMode is None:
        raise ImportError("Install arc-agi to use Toolkit / Arcade mode.")

    op_mode_name = "OFFLINE" if CFG.choose_mode() == "offline" else "ONLINE"
    op_mode = getattr(OperationMode, op_mode_name, getattr(OperationMode, "NORMAL", None))
    if op_mode is None:
        op_mode = OperationMode

    arc = Arcade(
        operation_mode=op_mode,
        arc_api_key=CFG.api_key,
        environments_dir=CFG.environments_dir,
        recordings_dir=CFG.recordings_dir,
        arc_base_url=CFG.base_url,
    )

    env = arc.make(game_id, save_recording=save_recording, render_mode=render_mode)
    if env is None:
        raise RuntimeError("Failed to create environment.")

    obs = env.reset()
    steps = 0

    while obs is not None and getattr(obs, "state", None) not in {"WIN", "GAME_OVER"} and steps < 50:
        actions = env.action_space
        available = [a.name if hasattr(a, "name") else a for a in actions]
        action_name = random.choice(available)

        action_obj = None
        if GameAction is not None and isinstance(action_name, str) and hasattr(GameAction, action_name):
            action_obj = getattr(GameAction, action_name)
        elif GameAction is not None:
            action_obj = getattr(GameAction, f"ACTION{random.randint(1, 7)}")
        else:
            action_obj = action_name

        if action_name == "ACTION6" and hasattr(action_obj, "is_complex") and action_obj.is_complex():
            obs = env.step(action_obj, data={"x": 32, "y": 32}, reasoning={"persona": "Wynt3r45"})
        else:
            obs = env.step(action_obj, reasoning={"persona": "Wynt3r45"})

        steps += 1

    scorecard = arc.get_scorecard()
    final_scorecard = arc.close_scorecard()
    return {
        "scorecard": scorecard,
        "final_scorecard": final_scorecard,
    }

# %% [code]
def render_games_examples(arc):
    """
    Render modes requested: terminal, terminal-fast, human, and custom renderer.
    """
    def custom_renderer(step: int, frame_data):
        print(f"Step {step}: state={getattr(frame_data, 'state', None)}")

    examples = {
        "terminal": arc.make("ls20", render_mode="terminal"),
        "terminal-fast": arc.make("ls20", render_mode="terminal-fast"),
        "human": arc.make("ls20", render_mode="human"),
        "custom": arc.make("ls20", renderer=custom_renderer),
    }
    return examples

# %% [code]
def listen_and_serve_example(arc):
    """
    Listen-and-serve support. Run only when you explicitly want to host the REST API locally.
    """
    if not hasattr(arc, "listen_and_serve"):
        return "listen_and_serve not available in this environment."

    def on_close(scorecard):
        print(f"Scorecard closed: {getattr(scorecard, 'score', None)}")

    def add_cookie(resp, api_key):
        # Example session-stickiness hook
        try:
            resp.set_cookie("APPLICATION_COOKIE", api_key, path="/", httponly=True)
        except Exception:
            pass
        return resp

    def register_custom(arcade, app):
        @app.route("/wynt3r45/status")
        def status():
            return {"ok": True, "mode": "serve"}

    return arc.listen_and_serve(
        port=8001,
        competition_mode=False,
        save_all_recordings=True,
        on_scorecard_close=on_close,
        add_cookie=add_cookie,
        extra_api_routes=register_custom,
        threaded=True,
        debug=False,
    )

# %% [code]
def competition_mode_example(api_key: str):
    """
    Competition mode: use API/Toolkit in the strict competition workflow.
    """
    if Arcade is None or OperationMode is None:
        raise ImportError("Install arc-agi to use competition mode.")
    comp_mode = getattr(OperationMode, "COMPETITION", getattr(OperationMode, "ONLINE", None))
    arc = Arcade(operation_mode=comp_mode, arc_api_key=api_key)
    return arc

# %% [code]
def benchmark_agent_api(game_ids: Optional[List[str]] = None, runs: int = 3):
    """
    Very small benchmarking hook.
    """
    if not CFG.api_key:
        raise RuntimeError("ARC_API_KEY required for online benchmarking.")
    client = ArcRestClient(CFG.api_key, CFG.base_url)
    games = client.list_games()
    chosen = game_ids or [g["game_id"] for g in games[:min(3, len(games))]]

    results = []
    for gid in chosen:
        for i in range(runs):
            card_id, _ = client.open_scorecard(tags=["benchmark", "wynt3r45"], opaque={"run": i})
            state = client.reset(gid, card_id)
            guid = state["guid"]
            steps = 0
            while state["state"] == "NOT_FINISHED" and steps < 10:
                action = PERSONA.choose_action(state.get("available_actions", []), state)
                if action == 6:
                    state = client.action(6, gid, guid, reasoning=PERSONA.build_reasoning({"benchmark": True}), x=32, y=32)
                else:
                    state = client.action(action, gid, guid, reasoning=PERSONA.build_reasoning({"benchmark": True}))
                steps += 1
            summary = client.close_scorecard(card_id)
            results.append({"game_id": gid, "run": i, "steps": steps, "summary": summary})
    return results

# %% [code]
def swarm_run_stub(game_ids: List[str]):
    """
    Swarm placeholder: run multiple agents across multiple games.
    Replace with threads/processes if needed.
    """
    out = []
    for gid in game_ids:
        try:
            out.append(run_api_full_playtest(game_id=gid, max_steps=25))
        except Exception as e:
            out.append({"game_id": gid, "error": str(e)})
    return out

# %% [code]
def create_own_agent_stub():
    """
    Replace Wynt3r45Persona with your own agent class here.
    """
    class YourAgent:
        def decide(self, obs):
            return 1
    return YourAgent()

# %% [code]
def feature_request_bug_notes():
    return {
        "bugs": "Open issue in ARC-AGI GitHub with label bug.",
        "feature_requests": "Open issue with label feature-request.",
        "questions": "Use question.",
        "docs": "Use documentation.",
        "contributing": "Keep code minimal, readable, and review AI-generated code carefully.",
    }

# %% [code]
def references_and_vocab():
    return {
        "vocabulary": {
            "ARC-AGI": "Abstraction and Reasoning Corpus for AGI",
            "ARC-AGI-3": "Interactive environments for exploring, learning, and adapting",
            "Arcade": "Main entry point class",
            "Environment": "Interactive game instance",
            "EnvironmentWrapper": "Common interface for local or remote environments",
            "Swarm": "Orchestrates multiple agents across multiple games",
            "Toolkit": "Python SDK for interacting with ARC-AGI-3",
        },
        "docs_pages": DOC_HINTS,
        "reference_notes": [
            "quickstart",
            "api-keys",
            "vocabulary",
            "create your own agent",
            "LLM Agents",
            "Partner templates: AgentOps, Anthropic, Hugging Face, LangChain",
            "local vs online",
            "scorecards",
            "recordings & replays",
            "full play test",
            "benchmarking",
            "swarms",
            "render games",
            "submit action",
            "feature requests & bugs",
            "contributing",
            "changelog",
        ],
    }

# %% [code]
print("Mode:", MODE)
print("Online available:", ONLINE_AVAILABLE)
print("API key present:", bool(CFG.api_key))
print("Docs preview:")
print(fetch_docs_index(online=ONLINE_AVAILABLE)[:1500])

# %% [code]
# Example usage:
# 1) Online API playtest:
# result = run_api_full_playtest(game_id=None, max_steps=25)
#
# 2) Toolkit mode:
# toolkit_result = run_toolkit_mode(game_id=DEFAULT_GAME, render_mode="terminal", save_recording=True)
#
# 3) Exact open-scorecard request:
# response = run_api_playtest_exactly_as_requested()
#
# 4) Kaggle offline input listing:
# print_kaggle_input_files()
#
# 5) Kaggle download paths:
# print(download_competition_data())
#
# 6) Internet heuristic:
# print(ping_pypi_heuristic())

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-